# LA Studio translation — M2M-100 418M

This notebook loads exactly `m2m100-418m` (`facebook/m2m100_418M`) on CUDA.
It is independent from API Gateway and rejects every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into the matching LA Studio feature.


In [ ]:
!nvidia-smi
%pip install -q "fastapi==0.115.12" "uvicorn==0.34.3" "transformers==4.57.3" "accelerate==1.12.0" "sentencepiece==0.2.1" "safetensors==0.6.2"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_translation_worker.py')
WORKER.write_text('import os\nimport re\nimport secrets\nimport threading\n\nimport torch\nfrom fastapi import Depends, FastAPI, Header, HTTPException\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.")\n\nfrom transformers import M2M100ForConditionalGeneration, M2M100Tokenizer\n\nMODEL_ID = "m2m100-418m"\nMODEL_NAME = "M2M-100 418M"\nUPSTREAM_MODEL = "facebook/m2m100_418M"\nUPSTREAM_REVISION = "55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636"\nSUPPORTED_LANGUAGES = "101 M2M100 language codes"\n\nTOKENIZER = M2M100Tokenizer.from_pretrained(UPSTREAM_MODEL, revision=UPSTREAM_REVISION)\nMODEL = M2M100ForConditionalGeneration.from_pretrained(\n    UPSTREAM_MODEL,\n    revision=UPSTREAM_REVISION,\n    torch_dtype=torch.float16,\n    low_cpu_mem_usage=True,\n).to("cuda").eval()\n\ndef translate_exact(texts: list[str], source: str, target: str) -> list[str]:\n    source = source.lower()\n    target = target.lower()\n    try:\n        TOKENIZER.src_lang = source\n        target_id = TOKENIZER.get_lang_id(target)\n    except KeyError as error:\n        raise HTTPException(status_code=422, detail=f"unsupported M2M100 language pair: {source} -> {target}") from error\n    def generate(batch: list[str], **generation_options: object) -> list[str]:\n        inputs = TOKENIZER(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")\n        with torch.inference_mode():\n            output = MODEL.generate(\n                **inputs,\n                forced_bos_token_id=target_id,\n                max_new_tokens=512,\n                **generation_options,\n            )\n        return TOKENIZER.batch_decode(output, skip_special_tokens=True)\n\n    translated = generate(texts)\n    # Greedy decoding can terminate at EOS immediately for a noisy but valid\n    # ASR segment. Retry only those blanks with the same pinned M2M checkpoint\n    # and a deterministic beam search; this is not a source-text fallback.\n    for index, value in enumerate(translated):\n        if isinstance(value, str) and value.strip():\n            continue\n        retry_text = " ".join(texts[index].split())\n        retry = generate(\n            [retry_text],\n            num_beams=4,\n            min_new_tokens=1,\n            early_stopping=True,\n            repetition_penalty=1.05,\n        )\n        translated[index] = retry[0] if retry else ""\n    return translated\n\nTOKEN = os.environ["LA_STUDIO_COLAB_TRANSLATION_TOKEN"]\nWORKER_REVISION = "translation-2026-07-30.3"\nRESPONSE_CONTRACT = "translation-patches-v3"\nMAX_TRANSLATION_SEGMENTS = 128\nMAX_TRANSLATION_CHARS = 50000\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\n\n# A translation result must never make the rest of a dubbing job disappear.\n# The worker retries blank model output once, then preserves the source in a\n# visible needs-review patch. This is deliberately not reported as a completed\n# translation: the editor can show it for review while later segments continue.\nNONLEXICAL_UTTERANCES = {\n    "ah", "aha", "eh", "er", "ha", "haha", "heh", "hmm", "hm", "ho", "oh",\n    "uh", "um", "wow", "嗯", "嗯哼", "啊", "啊哈", "哎", "哎呀", "诶", "欸",\n    "哈", "哈哈", "呵", "呵呵", "嘿", "哼", "唉", "呀", "呃", "哦", "哦哦",\n    "喔", "噢", "哇", "唔",\n}\n\ndef is_nonlexical_utterance(text: str) -> bool:\n    normalized = re.sub(r"[^\\w]", "", text, flags=re.UNICODE).casefold()\n    return normalized in NONLEXICAL_UTTERANCES\n\ndef retry_empty_translations(texts: list[str], translated: list[str], source: str, target: str) -> list[str]:\n    if not isinstance(translated, list) or len(translated) != len(texts):\n        return translated\n    empty_indices = [\n        index for index, value in enumerate(translated)\n        if not isinstance(value, str) or not value.strip()\n    ]\n    if not empty_indices:\n        return translated\n    # Retry only the affected source strings with the same selected, pinned\n    # model. If the retry itself fails, the patch builder below keeps the source\n    # and flags it for review instead of terminating unrelated segments.\n    try:\n        retry_values = translate_exact([texts[index] for index in empty_indices], source, target)\n    except Exception:\n        return translated\n    if not isinstance(retry_values, list) or len(retry_values) != len(empty_indices):\n        return translated\n    for index, retry_value in zip(empty_indices, retry_values):\n        if isinstance(retry_value, str) and retry_value.strip():\n            translated[index] = retry_value\n    return translated\n\ndef make_translation_patches(segments: list["TranslationSegment"], translated: list[str]) -> list[dict]:\n    if len(translated) != len(segments):\n        raise RuntimeError("model returned a different number of translations")\n    patches = []\n    for index, (item, value) in enumerate(zip(segments, translated), start=1):\n        target = value.strip() if isinstance(value, str) else ""\n        if target:\n            patches.append({"id": item.id, "targetText": target, "state": "translated"})\n            continue\n        source = item.sourceText.strip()\n        if is_nonlexical_utterance(source):\n            patches.append({\n                "id": item.id,\n                "targetText": source,\n                "state": "needs-review",\n                "translationDiagnostic": (\n                    "The exact translation model returned no lexical text for this short vocal reaction; "\n                    "the source was preserved for review."\n                ),\n            })\n            continue\n        patches.append({\n            "id": item.id,\n            "targetText": source,\n            "state": "needs-review",\n            "translationDiagnostic": (\n                "The exact translation model returned no text after retry; the source was preserved "\n                "and later segments continued. Review this segment before export."\n            ),\n        })\n    return patches\n\ndef authorize(authorization: str = Header(default="")):\n    if not secrets.compare_digest(authorization, "Bearer " + TOKEN):\n        raise HTTPException(status_code=401, detail="invalid or missing bearer token")\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. Open the notebook for the selected model.",\n        )\n\nclass TranslationSegment(BaseModel):\n    id: str = Field(min_length=1, max_length=128)\n    sourceText: str = Field(min_length=1, max_length=5000)\n\nclass TranslationRequest(BaseModel):\n    model: str\n    source_language: str = Field(min_length=2, max_length=12)\n    target_language: str = Field(min_length=2, max_length=12)\n    segments: list[TranslationSegment]\n\napp = FastAPI(title=f"LA Studio Translation - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(_: None = Depends(authorize)):\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "worker_revision": WORKER_REVISION,\n        "response_contract": RESPONSE_CONTRACT,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(_: None = Depends(authorize)):\n    return {\n        "contract_version": 1,\n        "response_contract": RESPONSE_CONTRACT,\n        "worker_revision": WORKER_REVISION,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "translation",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "device": "cuda",\n                "loaded": True,\n                "response_contract": RESPONSE_CONTRACT,\n            }],\n        }],\n    }\n\n@app.post("/v1/translations")\ndef translate(request: TranslationRequest, _: None = Depends(authorize)):\n    require_exact_model(request.model)\n    if not request.segments:\n        raise HTTPException(status_code=400, detail="segments must not be empty")\n    texts = [item.sourceText.strip() for item in request.segments]\n    if any(not text for text in texts):\n        raise HTTPException(status_code=400, detail="each segment needs sourceText")\n    if len(texts) > MAX_TRANSLATION_SEGMENTS or sum(map(len, texts)) > MAX_TRANSLATION_CHARS:\n        raise HTTPException(status_code=413, detail="translation request is too large")\n    if not INFERENCE_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="worker is busy; retry shortly")\n    try:\n        translated = translate_exact(\n            texts,\n            request.source_language.strip(),\n            request.target_language.strip(),\n        )\n        translated = retry_empty_translations(\n            texts,\n            translated,\n            request.source_language.strip(),\n            request.target_language.strip(),\n        )\n        return {"patches": make_translation_patches(request.segments, translated)}\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} translation failed: {type(error).__name__}: {str(error)[:300]}",\n        ) from error\n    finally:\n        INFERENCE_SLOTS.release()' + '\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'm2m100-418m'
# LA Studio worker launch contract: launch-2026-08-06.1
import json
import os
import queue
import re
import secrets
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'translation'
MODEL_ID = 'm2m100-418m'
PORT = 3943
TOKEN_ENV = 'LA_STUDIO_COLAB_TRANSLATION_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_TRANSLATION_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_TRANSLATION_MODEL'
WORKER_LOG = Path('/content/la_studio_translation_worker.log')
WORKER_MODULE = 'la_studio_translation_worker'
WORKER_PYTHON = sys.executable
WORKER_PYTHON_ISOLATED = False
WORKER_ENVIRONMENT = {}
REQUIRES_CUDA = True
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def process_cmdline(pid: int) -> str:
    """Read a Linux process command line without depending on psutil."""
    try:
        return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(
            "utf-8", errors="replace"
        ).strip()
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return ""


def all_processes():
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        pid = int(entry.name)
        command = process_cmdline(pid)
        if command:
            yield pid, command


def listening_processes(port: int) -> dict[int, str]:
    """Return PIDs listening on a local TCP port via /proc socket ownership."""
    target_port = f"{port:04X}"
    socket_inodes = set()
    for table_name in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(table_name).read_text(encoding="utf-8").splitlines()[1:]
        except FileNotFoundError:
            continue
        for line in lines:
            fields = line.split()
            if len(fields) < 10:
                continue
            local_address, state, inode = fields[1], fields[3], fields[9]
            if state == "0A" and local_address.rsplit(":", 1)[-1].upper() == target_port:
                socket_inodes.add(inode)
    if not socket_inodes:
        return {}

    listeners = {}
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        try:
            descriptors = (entry / "fd").iterdir()
        except (FileNotFoundError, PermissionError):
            continue
        for descriptor in descriptors:
            try:
                target = os.readlink(descriptor)
            except (FileNotFoundError, PermissionError, OSError):
                continue
            match = re.fullmatch(r"socket:\\[(\\d+)\\]", target)
            if match and match.group(1) in socket_inodes:
                pid = int(entry.name)
                listeners[pid] = process_cmdline(pid)
                break
    return listeners


def stop_pid(pid: int) -> None:
    if pid == os.getpid():
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.2)
    try:
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass


def reclaim_previous_la_studio_worker() -> None:
    """Stop only an older LA Studio worker/tunnel for this exact local port.

    Re-running a Colab cell keeps child processes alive.  The previous launch
    created a new token but aborted before it could replace the old worker,
    forcing users to destroy the whole GPU runtime.  We identify ownership by
    the exact generated module name and never terminate a foreign listener.
    """
    stopped = []
    for pid, command in listening_processes(PORT).items():
        if WORKER_MODULE in command and "uvicorn" in command:
            stop_pid(pid)
            stopped.append(f"worker PID {pid}")

    endpoint = f"http://127.0.0.1:{PORT}"
    for pid, command in all_processes():
        if ("cloudflared" in command and "tunnel" in command and endpoint in command):
            stop_pid(pid)
            stopped.append(f"tunnel PID {pid}")

    deadline = time.monotonic() + 12
    while port_is_occupied(PORT) and time.monotonic() < deadline:
        time.sleep(0.2)
    if stopped:
        print("Stopped previous LA Studio " + ", ".join(stopped) + ".")

    if port_is_occupied(PORT):
        listeners = listening_processes(PORT)
        foreign_pids = sorted(listeners) or ["unknown"]
        raise RuntimeError(
            f"Port {PORT} is occupied by a process that is not the previous LA Studio "
            f"{CAPABILITY_LABEL} worker (PID(s): {', '.join(map(str, foreign_pids))}). "
            "Choose a fresh Colab runtime rather than terminating an unrelated process."
        )


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


reclaim_previous_la_studio_worker()

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
env.update(WORKER_ENVIRONMENT)
if WORKER_PYTHON_ISOLATED:
    # Do not let Colab's global site-packages or a notebook-level PYTHONPATH
    # bleed into a dedicated worker virtual environment.
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [WORKER_PYTHON, "-m", "uvicorn", 'la_studio_translation_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    worker_kind = "exact CUDA" if REQUIRES_CUDA else "dedicated Colab CPU"
    print(f"Starting {worker_kind} {CAPABILITY_LABEL} worker.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower()
                        == ("cuda" if REQUIRES_CUDA else "colab-cpu")
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print(worker_kind.title() + " worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print(f"Waiting for the {worker_kind} worker...", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
